## Ingest Bronze-Orders Table
- Create an explicit spark Dataframe schema to read the data from landing
- 
- Ensure no Schema-Drift happens
- Ensure schema-evolution (New columns shouldn't cause problems)
- Some columns values can be null, PK can be duplicate etc. We need to keep the data readable and closer to source (raw) data

In [0]:
%run "../shared/00.common-variables"

In [0]:
%run "../shared/01.schemas"

In [0]:
%run "../shared/02.helper-functions"

In [0]:
%sql
show external locations

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType
from datetime import datetime
from pyspark.sql import functions as F
from zoneinfo import ZoneInfo



**Keeping all fields, nullable= True**

Because later, when discussing architecture you can say:

Bronze prioritizes ingestion reliability over business correctness.
Hence, All columns are nullable.

And, Business Data Quality Check is applied in Silver.

## Ingestion Pipeline

In [0]:
# %sql
# -- FOR TESTING PURPOSES ONLY: Comment after running
# TRUNCATE TABLE olist.bronze.orders;
# TRUNCATE TABLE olist.logs.pipeline_logs;

In [0]:
## ---------------------------Config Variables-------------------------------------------------------------------

start_time = datetime.now(ZoneInfo("Asia/Kolkata"))
end_time = None

table_name = "orders"
business_key = 'order_id'
full_table_name = f"{catalog_name}.{bronze_schema}.{table_name}"
source_file_path = f"{landing_folder_path}/olist_orders_dataset.csv"
file_name = 'olist_orders_dataset.csv'

pipeline_name = f"{bronze_schema}_{table_name}"
pipeline_run_id = generate_pipeline_run_id(pipeline_name, start_time)


## ---------------------------Pipeline Run-------------------------------------------------------------------


source_row_count = None
target_row_count = None
null_business_key_count = None
duplicated_business_key_count = None
corrupt_row_count = None
status = None
error_message = None

pipeline_exception = None

#----------- Pipeline Control Variables ----------#

file_checksum = None
retry_count = 0
file_location = None

try:

    file_checksum = generate_file_checksum(source_file_path)

    if is_file_already_processed(file_checksum, full_pipeline_control_table_name):
        status = "SKIPPED_DUPLICATE"
        error_message = "File already processed"

        # Move file to duplicates/ folder
        target_file_path = f"{duplicates_folder_path}/{file_name}"
        dbutils.fs.mv(source_file_path, target_file_path)
        file_location = target_file_path


    else:

        orders_df = spark.read.format('csv')\
                        .option("header",True)\
                        .option("mode","PERMISSIVE")\
                        .option("columnNameOfCorruptRecord", "_corrupt_record")\
                        .schema(orders_schema)\
                        .load(source_file_path)


        orders_df = add_ingestion_metadata(orders_df, pipeline_run_id)

        source_row_count, target_row_count, null_business_key_count,\
        duplicated_business_key_count, corrupt_row_count = calculate_dq_metrics(orders_df,business_key)

        # target_row_count = source_row_count 
        # Because no transformation, deduplication etc. being done on source data. and writing count() again would trigger an action which could be expensive with large data
        

        orders_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(full_table_name)


        status = "SUCCESS"
        error_message = None

        # Move file to processed/ folder
        target_file_path = f"{processed_folder_path}/{file_name}"
        dbutils.fs.mv(source_file_path, target_file_path)
        file_location = target_file_path
    

except Exception as e:
    pipeline_exception = e
    error_message = str(e)
    status = "FAILED"

    # Move file to failed/ folder
    target_file_path = f"{failed_folder_path}/{file_name}"
    dbutils.fs.mv(source_file_path, target_file_path)
    file_location = target_file_path


finally:
    end_time = datetime.now(ZoneInfo(my_zone))    
    try:

        pipeline_control_data = [(
        file_checksum,
        file_name,
        source_file_path,
        pipeline_name,
        pipeline_run_id,
        table_name,
        status,
        retry_count,
        start_time,
        end_time,
        source_row_count,
        target_row_count,
        error_message,
        file_location
        )]

        pipeline_control_df = spark.createDataFrame(
            pipeline_control_data,
            schema=pipeline_control_schema
        )

        update_pipeline_control(pipeline_control_df, full_pipeline_control_table_name)
          
        logs_data = [
        (
            pipeline_run_id,
            pipeline_name,
            table_name,
            start_time,
            end_time,
            status,
            source_row_count,
            target_row_count,
            null_business_key_count,
            duplicated_business_key_count,
            corrupt_row_count,
            error_message
        )
        ]

        logs_df = spark.createDataFrame(
            logs_data,
            schema=logs_df_schema
        )

        write_logs(logs_df, full_logs_table_name)
        
        if pipeline_exception:
            raise pipeline_exception


    except Exception as logging_error:
        print(f"logging FAILED: {logging_error}")
        raise # This will make the notebook fail. If we don't use this, the message is just printed and notebook run continues.Which we don't want.



In [0]:
%sql
select * from olist.logs.pipeline_logs

## For Reference

### Preparing Dataframe : current-table_df

In [0]:


# orders_df = spark.read.format('csv')\
#                  .option("header",True)\
#                  .option("mode","PERMISSIVE")\
#                  .option("columnNameOfCorruptRecord", "_corrupt_record")\
#                  .schema(orders_schema)\
#                  .load(source_file_path)


# orders_df = add_ingestion_metadata(orders_df, pipeline_run_id)





In [0]:
# display(orders_df.limit(20))

In [0]:
# orders_df.printSchema()

In [0]:
# source_row_count, target_row_count, null_order_id_count,\
#     duplicated_order_id_count, corrupt_row_count = calculate_dq_metrics(orders_df,"order_id")

# # target_row_count = source_row_count 
# # Because no transformation, deduplication etc. being done on source data. and writing count() again would trigger an action which could be expensive with large data

# end_time = datetime.now(ZoneInfo(my_zone))
# status = "SUCCESS"
# error_message = None

### Create Table: current_table

In [0]:
# spark.sql(f"""CREATE TABLE IF NOT EXISTS {full_table_name} 
# (
#     order_id                         STRING,
#     order_status                     STRING,
#     customer_id                      STRING,
#     order_purchase_timestamp         TIMESTAMP,
#     order_approved_at                TIMESTAMP,
#     order_delivered_carrier_date     TIMESTAMP,
#     order_delivered_customer_date    TIMESTAMP,
#     order_estimated_delivery_date    DATE,
#     _corrupt_record                  STRING,
#     ingestion_timestamp              TIMESTAMP NOT NULL,
#     source_file_name                 STRING NOT NULL,
#     source_system                    STRING NOT NULL,
#     pipeline_run_id                  STRING NOT NULL
# )
# USING DELTA
# LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/orders"
# """)

In [0]:
# orders_df.write \
#     .format("delta") \
#     .mode("append") \
#     .saveAsTable(full_table_name)

### Logging

In [0]:
# logs_data = [
#     (
#         pipeline_run_id,
#         pipeline_name,
#         table_name,
#         start_time,
#         end_time,
#         status,
#         source_row_count,
#         target_row_count,
#         null_order_id_count,
#         duplicated_order_id_count,
#         corrupt_row_count,
#         error_message
#     )
# ]

# logs_df = spark.createDataFrame(
#     logs_data,
#     schema=logs_df_schema
# )

In [0]:
# display(logs_df)

In [0]:
# write_logs(logs_df, full_logs_table_name)